## Currency Converter

In [71]:
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

import requests

In [72]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

# tool

@tool
def get_convesrion_factor(base_currency: str, target_currency: str) -> float:
    """
        This function fetches the currency conversion factor between the given base currency and the target currency.
    """

    url = f"https://v6.exchangerate-api.com/v6/4fbf9973bd4275857522a93b/pair/{base_currency}/{target_currency}"
    response = requests.get(url)

    return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
        Given a currency conversion rate. This function calculates the target currency value from a base currency value
    """

    return base_currency_value *  conversion_rate

In [73]:
get_convesrion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1794528001,
 'time_last_update_utc': 'Fri, 13 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1794614401,
 'time_next_update_utc': 'Sat, 14 Nov 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.0857}

In [74]:
convert.invoke({"base_currency_value": 10, "conversion_rate": 94.0857})

940.857

In [75]:
### tool binding

llm = ChatGroq(model="openai/gpt-oss-120b")

In [76]:
llm_with_tools = llm.bind_tools([get_convesrion_factor, convert])

In [77]:
messages = [HumanMessage("What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr")]

In [78]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [79]:
ai_message = llm_with_tools.invoke(messages)

In [80]:
ai_message

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need conversion factor between INR and USD. Use function get_convesrion_factor with base_currency "USD", target_currency "INR" perhaps. Then convert 10 USD to INR using convert function? Actually convert function takes base_currency_value and presumably uses the previously fetched conversion factor? The description: "Given a currency conversion rate. This function calculates the target currency value from a base currency value". It expects base_currency_value and maybe also conversion factor? But description says "Given a currency conversion rate." Might need to pass conversion factor? The function signature only has base_currency_value. Possibly the tool uses a previously set conversion factor? Not sure. Maybe we need to call get_convesrion_factor to get factor, then compute manually: INR = USD * factor. Could also call convert with base_currency_value = 10 and maybe the function uses default conversion? Might need to p

In [81]:
messages.append(ai_message)

In [82]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need conversion factor between INR and USD. Use function get_convesrion_factor with base_currency "USD", target_currency "INR" perhaps. Then convert 10 USD to INR using convert function? Actually convert function takes base_currency_value and presumably uses the previously fetched conversion factor? The description: "Given a currency conversion rate. This function calculates the target currency value from a base currency value". It expects base_currency_value and maybe also conversion factor? But description says "Given a currency conversion rate." Might need to pass conversion factor? The function signature only has base_currency_value. Possibly the tool uses a previously set conversion factor? Not sure. Maybe we need to call get_convesrion_factor to ge

In [83]:
ai_message.tool_calls

[{'name': 'get_convesrion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'fc_8bdc793b-457a-4cc7-a0ce-f5c2fd60d48e',
  'type': 'tool_call'}]

In [84]:

import json

for tool_call in ai_message.tool_calls:

    if tool_call["name"] == "gget_convesrion_factor":

        tool_msg1 = get_convesrion_factor.invoke(
            tool_call["args"]
        )

        conversion_rate = tool_msg1["conversion_rate"]

        messages.append(tool_msg1)

        # Now execute the second tool manually
        tool_msg2 = convert.invoke({
            "base_currency_value": 10,
            "conversion_rate": conversion_rate
        })

        messages.append(tool_msg2)


In [85]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need conversion factor between INR and USD. Use function get_convesrion_factor with base_currency "USD", target_currency "INR" perhaps. Then convert 10 USD to INR using convert function? Actually convert function takes base_currency_value and presumably uses the previously fetched conversion factor? The description: "Given a currency conversion rate. This function calculates the target currency value from a base currency value". It expects base_currency_value and maybe also conversion factor? But description says "Given a currency conversion rate." Might need to pass conversion factor? The function signature only has base_currency_value. Possibly the tool uses a previously set conversion factor? Not sure. Maybe we need to call get_convesrion_factor to ge

In [87]:
response = llm_with_tools.invoke(messages)
print(response.content)

In [88]:
response.content

''